# Banking Fraud Intelligence
## P5 · Power BI data preparation and quality assurance

**Notebook 05 | Reporting layer | Python 3.12 | pandas**

This notebook prepares the **approved reporting tables** for the Power BI investigation dashboard. It is the bridge between the completed P4 evaluation and the Power BI semantic model.

### Business objective

Prepare a clean, auditable reporting layer that supports the two approved report pages:

1. **Fraud Overview:** daily transaction activity, retrospective fraud-label prevalence, monthly summaries and simulator-rule context.
2. **Review Policy Comparison:** September held-out ranks, equal-capacity policy comparisons and the fixed four-scenario reference matrix.

### What this notebook does

- Re-validates the existing full transaction CSV without changing source rows.
- Aggregates 1,754,155 transactions into a daily reporting table.
- Creates a matching 183-day date dimension.
- Re-validates the P4 September rank export and all eight policy/cohort/capacity comparisons.
- Creates a compact Power BI import table and a fixed September scenario-lift table.
- Displays visual quality checks before exporting the local reporting files.

> **Data integrity:** This notebook does not retrain the model, change the frozen evaluation protocol, modify source transactions, or publish row-level data. All generated CSVs remain under the Git-ignored data/processed/ directory.

## Run contract

**Prerequisites**

- data/processed/transactions.csv must exist and pass the existing input validator.
- data/processed/september_review_ranks.csv must exist. It is generated by Notebook 04.
- Use the same Python 3.12 environment used for the earlier notebooks.

**How to run**

Open this notebook from the repository checkout and choose **Run All**. If you want to preserve the executed notebook and charts, first copy it into data/processed/notebook-runs/ using the workflow in data/README.md.

**Outputs**

All outputs are generated locally in data/processed/powerbi/. The generated files are ignored by Git and must not be committed.

| Output | Grain | Intended use |
|---|---|---|
| Overview_Daily.csv | One row per day, 183 rows | Fraud Overview page |
| DimDate.csv | One row per day, 183 rows | Shared date dimension |
| September_Ranks.csv | One row per September transaction | Review Policy Comparison page |
| September_Scenario_Lift.csv | Four cohort/capacity combinations | Fixed September reference matrix |
| September_Queue_Reconciliation_QA.csv | Eight policy/cohort/capacity checks | QA evidence only; do not load into the report |
| PowerBI_Data_Manifest.json | One manifest | Input/output hashes and row-count evidence

In [ ]:
from datetime import datetime, timezone
from hashlib import file_digest
import json
from pathlib import Path
import sys

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib.ticker import PercentFormatter

REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "validate_transactions.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook from the Banking Fraud Intelligence repository checkout."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.convert_transactions import OUTPUT as TRANSACTIONS_CSV
from src.validate_transactions import load_transactions

RANK_SOURCE = TRANSACTIONS_CSV.with_name("september_review_ranks.csv")
OUTPUT_DIR = REPO_ROOT / "data" / "processed" / "powerbi"

PALETTE = {
    "navy": "#0F172A",
    "baseline": "#1E3A5F",
    "model": "#0F766E",
    "secondary": "#475569",
    "muted": "#EEF2F7",
    "border": "#CBD5E1",
}

plt.rcParams.update({
    "figure.facecolor": "#F8FAFC",
    "axes.facecolor": "#FFFFFF",
    "axes.edgecolor": "#CBD5E1",
    "axes.labelcolor": "#475569",
    "axes.titlecolor": "#0F172A",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": False,
    "axes.spines.bottom": False,
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "grid.color": "#E2E8F0",
    "xtick.color": "#475569",
    "ytick.color": "#475569",
})

def sha256_file(path: Path) -> str:
    with path.open("rb") as handle:
        return file_digest(handle, "sha256").hexdigest()

print(f"Repository: {REPO_ROOT}")
print(f"Transactions input: {TRANSACTIONS_CSV.relative_to(REPO_ROOT)}")
print(f"September rank input: {RANK_SOURCE.relative_to(REPO_ROOT)}")
print(f"Power BI output folder: {OUTPUT_DIR.relative_to(REPO_ROOT)}")

## 01 · Load and validate the source inputs

**Question:** Are the full transaction CSV and September rank export present and consistent with the audited P1–P4 source?

The full CSV is passed through the existing project validator. The September export is checked for its expected grain, date coverage, unique transaction IDs and the four rank columns required by the approved report.

No records are dropped or corrected in this reporting-preparation step.

In [ ]:
if not TRANSACTIONS_CSV.is_file():
    raise FileNotFoundError(
        "The validated transaction CSV is missing. Follow data/README.md#prepare-one-csv."
    )

if not RANK_SOURCE.is_file():
    raise FileNotFoundError(
        "The September rank export is missing. Run notebooks/04_fraud_holdout_evaluation.ipynb first."
    )

transactions_sha256 = sha256_file(TRANSACTIONS_CSV)
rank_source_sha256 = sha256_file(RANK_SOURCE)

transactions = load_transactions(TRANSACTIONS_CSV)

assert len(transactions) == 1_754_155
assert int(transactions["TX_FRAUD"].sum()) == 14_681
assert transactions["TRANSACTION_ID"].is_unique
assert transactions["TX_DATETIME"].is_monotonic_increasing
assert not transactions.isna().any().any()

expected_dates = pd.date_range("2018-04-01", "2018-09-30", freq="D")
observed_dates = pd.DatetimeIndex(transactions["TX_DATETIME"].dt.normalize().unique())
assert observed_dates.equals(expected_dates)

above_220 = transactions["TX_AMOUNT"].gt(220)
assert int(above_220.sum()) == 3_258
assert transactions.loc[above_220, "TX_FRAUD"].eq(1).all()
assert int(transactions.loc[~above_220, "TX_FRAUD"].sum()) == 11_423

input_audit = pd.DataFrame([
    {"Check": "Transactions", "Observed": f"{len(transactions):,}", "Expected": "1,754,155"},
    {"Check": "Fraud-labelled transactions", "Observed": f"{int(transactions['TX_FRAUD'].sum()):,}", "Expected": "14,681"},
    {"Check": "Date coverage", "Observed": f"{observed_dates.min():%d %b %Y} to {observed_dates.max():%d %b %Y}", "Expected": "01 Apr 2018 to 30 Sep 2018"},
    {"Check": "Distinct dates", "Observed": f"{observed_dates.nunique():,}", "Expected": "183"},
    {"Check": "Transactions above 220", "Observed": f"{int(above_220.sum()):,}", "Expected": "3,258"},
    {"Check": "Fraud labels at or below 220", "Observed": f"{int(transactions.loc[~above_220, 'TX_FRAUD'].sum()):,}", "Expected": "11,423"},
])
display(input_audit)

print("PASS · Existing source validator accepted all rows. No source values were changed.")

## 02 · Build the daily Fraud Overview table

The report needs daily counts so that Power BI can calculate correct period-level and weekly ratios.

**Important modelling decision:** fraud-labelled share is not exported as an additive daily value. Power BI will calculate it as:

\[
\\text{Fraud-labelled share} =
\\frac{\\sum \\text{Fraud-labelled transactions}}
{\\sum \\text{Transactions}}
\]

This preserves a correctly weighted ratio when the user selects a week, month or custom date range.

In [ ]:
overview_input = transactions.assign(
    Date=transactions["TX_DATETIME"].dt.normalize(),
    FraudLabelledValue=transactions["TX_AMOUNT"].where(transactions["TX_FRAUD"].eq(1), 0.0),
    Above220Transaction=transactions["TX_AMOUNT"].gt(220),
    Above220FraudLabelled=transactions["TX_FRAUD"].where(transactions["TX_AMOUNT"].gt(220), 0),
    AtOrBelow220Transaction=transactions["TX_AMOUNT"].le(220),
    AtOrBelow220FraudLabelled=transactions["TX_FRAUD"].where(transactions["TX_AMOUNT"].le(220), 0),
)

overview_daily = (
    overview_input.groupby("Date", as_index=False, sort=True)
    .agg(
        TransactionCount=("TRANSACTION_ID", "size"),
        FraudLabelledCount=("TX_FRAUD", "sum"),
        FraudLabelledTransactionValue=("FraudLabelledValue", "sum"),
        Above220TransactionCount=("Above220Transaction", "sum"),
        Above220FraudLabelledCount=("Above220FraudLabelled", "sum"),
        AtOrBelow220TransactionCount=("AtOrBelow220Transaction", "sum"),
        AtOrBelow220FraudLabelledCount=("AtOrBelow220FraudLabelled", "sum"),
    )
)

assert len(overview_daily) == 183
assert overview_daily["Date"].is_unique
assert int(overview_daily["TransactionCount"].sum()) == 1_754_155
assert int(overview_daily["FraudLabelledCount"].sum()) == 14_681
assert int(overview_daily["Above220FraudLabelledCount"].sum()) == 3_258
assert int(overview_daily["AtOrBelow220FraudLabelledCount"].sum()) == 11_423
assert overview_daily["Date"].min() == pd.Timestamp("2018-04-01")
assert overview_daily["Date"].max() == pd.Timestamp("2018-09-30")

display(overview_daily.head(8))
print(
    f"PASS · {len(overview_daily):,} daily rows reconcile to "
    f"{int(overview_daily['TransactionCount'].sum()):,} transactions and "
    f"{int(overview_daily['FraudLabelledCount'].sum()):,} fraud labels."
)

## 03 · Build the shared date dimension

The date dimension supports consistent date filtering, month grouping and weekly aggregation on both report pages.

Weeks start on Monday. The first reporting week contains only 1 April 2018, so it is a **partial week**. That is retained and explicitly noted in the visual quality check rather than silently dropping a day.

In [ ]:
dim_date = pd.DataFrame({"Date": expected_dates})
dim_date["Year"] = dim_date["Date"].dt.year
dim_date["MonthNumber"] = dim_date["Date"].dt.month
dim_date["MonthName"] = dim_date["Date"].dt.month_name()
dim_date["MonthYear"] = dim_date["Date"].dt.strftime("%b %Y")
dim_date["MonthStartDate"] = dim_date["Date"].dt.to_period("M").dt.to_timestamp()
dim_date["WeekStartDate"] = dim_date["Date"] - pd.to_timedelta(dim_date["Date"].dt.weekday, unit="D")
dim_date["DayOfWeekNumber"] = dim_date["Date"].dt.dayofweek + 1
dim_date["DayName"] = dim_date["Date"].dt.day_name()
dim_date["IsWeekend"] = dim_date["Date"].dt.dayofweek.ge(5)

assert len(dim_date) == 183
assert dim_date["Date"].is_unique
assert dim_date["Date"].equals(overview_daily["Date"])
assert dim_date["MonthNumber"].between(4, 9).all()

display(dim_date.head(7))
print("PASS · Date dimension covers every source date exactly once.")

## 04 · Inspect the temporal pattern and simulator context

Before loading the aggregate, inspect the weekly and monthly summaries. The weekly share is calculated from summed fraud labels divided by summed transactions, not by averaging daily percentages.

The amount boundary is a simulator rule. It is shown as context for the sensitivity cohort, not as a proposed real-world banking threshold.

In [ ]:
overview_with_calendar = overview_daily.merge(
    dim_date[["Date", "MonthStartDate", "MonthYear", "WeekStartDate"]],
    on="Date",
    how="left",
    validate="one_to_one",
)

weekly_summary = (
    overview_with_calendar.groupby("WeekStartDate", as_index=False, sort=True)
    .agg(
        DaysCovered=("Date", "size"),
        TransactionCount=("TransactionCount", "sum"),
        FraudLabelledCount=("FraudLabelledCount", "sum"),
    )
)
weekly_summary["FraudLabelledShare"] = (
    weekly_summary["FraudLabelledCount"] / weekly_summary["TransactionCount"]
)

monthly_summary = (
    overview_with_calendar.groupby(["MonthStartDate", "MonthYear"], as_index=False, sort=True)
    .agg(
        TransactionCount=("TransactionCount", "sum"),
        FraudLabelledCount=("FraudLabelledCount", "sum"),
        FraudLabelledTransactionValue=("FraudLabelledTransactionValue", "sum"),
    )
)
monthly_summary["FraudLabelledShare"] = (
    monthly_summary["FraudLabelledCount"] / monthly_summary["TransactionCount"]
)

display(monthly_summary.style.format({
    "TransactionCount": "{:,.0f}",
    "FraudLabelledCount": "{:,.0f}",
    "FraudLabelledTransactionValue": "{:,.2f}",
    "FraudLabelledShare": "{:.3%}",
}))

assert len(monthly_summary) == 6
assert int(monthly_summary["TransactionCount"].sum()) == 1_754_155
assert int(monthly_summary["FraudLabelledCount"].sum()) == 14_681
assert len(weekly_summary) == 27
assert int(weekly_summary.iloc[0]["DaysCovered"]) == 1

print("PASS · Six monthly groups and 27 Monday-start weekly groups reconcile. The first week is a one-day partial week.")

In [ ]:
overall_share = overview_daily["FraudLabelledCount"].sum() / overview_daily["TransactionCount"].sum()

figure, axis = plt.subplots(figsize=(12, 4.2), layout="constrained")
axis.plot(
    weekly_summary["WeekStartDate"],
    weekly_summary["FraudLabelledShare"],
    color=PALETTE["model"],
    linewidth=2.5,
    marker="o",
    markersize=4.5,
    label="Weekly fraud-labelled share",
)
axis.axhline(
    overall_share,
    color=PALETTE["baseline"],
    linewidth=1.3,
    linestyle="--",
    label=f"Full-period share · {overall_share:.3%}",
)
axis.set_title("Fraud-labelled share over time", loc="left", fontsize=14, fontweight="semibold", pad=14)
axis.set_xlabel("Week starting")
axis.set_ylabel("Fraud-labelled share")
axis.yaxis.set_major_formatter(PercentFormatter(1.0))
axis.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO, interval=3))
axis.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
axis.grid(axis="y", linewidth=0.8)
axis.legend(frameon=False, loc="upper right")
axis.text(
    0.01, -0.24,
    "Note: the first point covers 1 April only; subsequent points cover full Monday–Sunday weeks.",
    transform=axis.transAxes,
    color=PALETTE["secondary"],
    fontsize=9,
)
plt.show()

threshold_summary = pd.DataFrame({
    "Cohort": ["Amounts above 220", "Amounts at or below 220"],
    "FraudLabelledTransactions": [
        int(overview_daily["Above220FraudLabelledCount"].sum()),
        int(overview_daily["AtOrBelow220FraudLabelledCount"].sum()),
    ],
})
figure, axis = plt.subplots(figsize=(9, 2.8), layout="constrained")
bars = axis.barh(
    threshold_summary["Cohort"],
    threshold_summary["FraudLabelledTransactions"],
    color=[PALETTE["baseline"], PALETTE["model"]],
    height=0.55,
)
axis.bar_label(bars, labels=[f"{value:,.0f}" for value in threshold_summary["FraudLabelledTransactions"]], padding=7, fontsize=10)
axis.set_title("Where the simulator's fraud labels occur", loc="left", fontsize=13, fontweight="semibold", pad=12)
axis.set_xlabel("Fraud-labelled transactions")
axis.set_xlim(0, threshold_summary["FraudLabelledTransactions"].max() * 1.18)
axis.grid(axis="x", linewidth=0.8)
axis.set_axisbelow(True)
plt.show()

display(threshold_summary)
print("Interpretation: all transactions above 220 are simulator-labelled fraud; the restricted cohort is a sensitivity check, not a production rule.")

## 05 · Validate the September held-out rank export

**Question:** Does the P4 export still reproduce the eight approved comparisons?

The validation checks:
- one row per September transaction;
- all 30 September dates;
- unique transaction IDs;
- complete full-population ranks;
- restricted ranks only for transactions at or below 220;
- unique, sequential ranks within each day;
- exactly 50 or 100 selected reviews per day;
- the documented fraud-labelled counts for all eight comparisons.

This is a reporting-data integrity check. It does not retrain or rescore the model.

In [ ]:
rank_export = pd.read_csv(
    RANK_SOURCE,
    parse_dates=["TX_DATETIME"],
    float_precision="round_trip",
)

RANK_COLUMNS = {
    ("All transactions", "Amount baseline"): "tx_amount_rank_all",
    ("All transactions", "Logistic regression"): "model_score_rank_all",
    ("Amounts at or below 220", "Amount baseline"): "tx_amount_rank_le_220",
    ("Amounts at or below 220", "Logistic regression"): "model_score_rank_le_220",
}
required_rank_fields = {
    "TRANSACTION_ID", "TX_DATETIME", "TX_AMOUNT", "TX_FRAUD",
    "model_score", *RANK_COLUMNS.values(),
}
missing_rank_fields = required_rank_fields.difference(rank_export.columns)
if missing_rank_fields:
    raise ValueError(f"September rank export is missing fields: {sorted(missing_rank_fields)}")

assert len(rank_export) == 287_873
assert rank_export["TRANSACTION_ID"].is_unique
assert rank_export["TX_DATETIME"].dt.normalize().min() == pd.Timestamp("2018-09-01")
assert rank_export["TX_DATETIME"].dt.normalize().max() == pd.Timestamp("2018-09-30")
assert rank_export["TX_DATETIME"].dt.normalize().nunique() == 30
assert rank_export["TX_AMOUNT"].ge(0).all()
assert rank_export["TX_FRAUD"].isin([0, 1]).all()

cohort_masks = {
    "All transactions": pd.Series(True, index=rank_export.index),
    "Amounts at or below 220": rank_export["TX_AMOUNT"].le(220),
}

for (cohort, _policy), column in RANK_COLUMNS.items():
    eligible = cohort_masks[cohort]
    assert rank_export[column].notna().equals(eligible), (
        f"Unexpected null pattern in {column}."
    )
    ranked = rank_export.loc[eligible, ["TX_DATETIME", column]].copy()
    ranked["Date"] = ranked["TX_DATETIME"].dt.normalize()
    by_day = ranked.groupby("Date")[column]
    assert by_day.min().eq(1).all()
    assert by_day.max().eq(by_day.size()).all()
    assert by_day.nunique().eq(by_day.size()).all()

print("PASS · September export has 287,873 unique transactions, 30 dates and four valid daily-rank columns.")

In [ ]:
EXPECTED_FRAUD_SELECTED = {
    ("All transactions", 50, "Amount baseline"): 589,
    ("All transactions", 50, "Logistic regression"): 592,
    ("All transactions", 100, "Amount baseline"): 622,
    ("All transactions", 100, "Logistic regression"): 632,
    ("Amounts at or below 220", 50, "Amount baseline"): 74,
    ("Amounts at or below 220", 50, "Logistic regression"): 77,
    ("Amounts at or below 220", 100, "Amount baseline"): 101,
    ("Amounts at or below 220", 100, "Logistic regression"): 110,
}

reconciliation_rows = []

for cohort, eligible in cohort_masks.items():
    cohort_fraud_total = int(rank_export.loc[eligible, "TX_FRAUD"].sum())

    for capacity in (50, 100):
        for policy in ("Amount baseline", "Logistic regression"):
            rank_column = RANK_COLUMNS[(cohort, policy)]
            selected = rank_export.loc[eligible & rank_export[rank_column].le(capacity)].copy()

            reviews = len(selected)
            fraud_selected = int(selected["TX_FRAUD"].sum())
            legitimate_reviews = reviews - fraud_selected
            selected_by_day = selected.groupby(
                selected["TX_DATETIME"].dt.normalize()
            ).size()

            assert len(selected_by_day) == 30
            assert selected_by_day.eq(capacity).all()
            assert reviews == 30 * capacity

            expected_count = EXPECTED_FRAUD_SELECTED[(cohort, capacity, policy)]
            assert fraud_selected == expected_count, (
                f"{cohort}, {capacity}/day, {policy}: "
                f"expected {expected_count}, observed {fraud_selected}."
            )

            fraud_value = float(
                selected.loc[selected["TX_FRAUD"].eq(1), "TX_AMOUNT"].sum()
            )

            reconciliation_rows.append({
                "Cohort": cohort,
                "DailyCapacity": capacity,
                "Policy": policy,
                "ReviewsSelected": reviews,
                "FraudLabelledSelected": fraud_selected,
                "LegitimateReviews": legitimate_reviews,
                "QueuePrecision": fraud_selected / reviews,
                "CohortRecall": fraud_selected / cohort_fraud_total,
                "FraudLabelledValueSelected": fraud_value,
            })

queue_reconciliation = pd.DataFrame(reconciliation_rows)

display(queue_reconciliation.style.format({
    "ReviewsSelected": "{:,.0f}",
    "FraudLabelledSelected": "{:,.0f}",
    "LegitimateReviews": "{:,.0f}",
    "QueuePrecision": "{:.2%}",
    "CohortRecall": "{:.2%}",
    "FraudLabelledValueSelected": "{:,.2f}",
}))

assert len(queue_reconciliation) == 8
print("PASS · All eight September policy/cohort/capacity comparisons reconcile to the recorded P4 findings.")

## 06 · Build the approved Power BI import tables

Four tables are prepared for the semantic model:

1. Overview_Daily
2. DimDate
3. September_Ranks
4. September_Scenario_Lift

The eight-row reconciliation table is retained as a local QA artifact and is **not** intended for import into the report.

The model score is not imported because the report uses the verified daily rank columns, not a probability display. The transaction ID remains available for uniqueness checks but should be hidden in the Power BI report view. Transaction-level drillthrough is not part of the approved first release.

In [ ]:
scenario_lift = (
    queue_reconciliation.pivot(
        index=["Cohort", "DailyCapacity"],
        columns="Policy",
        values="FraudLabelledSelected",
    )
    .reset_index()
)
scenario_lift.columns.name = None
scenario_lift["ModelDifference"] = (
    scenario_lift["Logistic regression"] - scenario_lift["Amount baseline"]
)
scenario_lift = scenario_lift.rename(columns={
    "Amount baseline": "AmountBaselineFraudLabelledSelected",
    "Logistic regression": "LogisticRegressionFraudLabelledSelected",
})
scenario_lift = scenario_lift.sort_values(
    ["Cohort", "DailyCapacity"], kind="stable"
).reset_index(drop=True)

september_powerbi = rank_export[
    [
        "TRANSACTION_ID", "TX_AMOUNT", "TX_FRAUD",
        "tx_amount_rank_all", "model_score_rank_all",
        "tx_amount_rank_le_220", "model_score_rank_le_220",
    ]
].copy()
september_powerbi.insert(
    1, "Date", rank_export["TX_DATETIME"].dt.normalize()
)

assert len(scenario_lift) == 4
assert scenario_lift["ModelDifference"].tolist() == [3, 10, 3, 9]
assert len(september_powerbi) == 287_873
assert september_powerbi["TRANSACTION_ID"].is_unique
assert september_powerbi["Date"].dt.normalize().nunique() == 30

display(scenario_lift)
display(september_powerbi.head(5))
print("PASS · Import tables have the approved grains and fields.")

## 07 · Export local Power BI data and create the manifest

The output folder is inside data/processed/, which is already excluded by the repository's .gitignore.

The manifest records the exact input fingerprints and the row count, schema and SHA-256 hash for every output. This gives us a reproducible checkpoint when the tables are loaded into Power BI Desktop.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def write_csv_atomic(frame: pd.DataFrame, path: Path) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    try:
        frame.to_csv(temporary, index=False, date_format="%Y-%m-%d")
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)

output_frames = {
    "Overview_Daily.csv": overview_daily,
    "DimDate.csv": dim_date,
    "September_Ranks.csv": september_powerbi,
    "September_Scenario_Lift.csv": scenario_lift,
    "September_Queue_Reconciliation_QA.csv": queue_reconciliation,
}

for filename, frame in output_frames.items():
    write_csv_atomic(frame, OUTPUT_DIR / filename)

output_manifest = {
    "project": "Banking Fraud Intelligence",
    "dataset_revision": "6e67dbd0a3bfe0d7ec33abc4bce5f37cd4ff0d6a",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "inputs": {
        "transactions_csv": {
            "path": str(TRANSACTIONS_CSV.relative_to(REPO_ROOT)),
            "rows": len(transactions),
            "sha256": transactions_sha256,
        },
        "september_review_ranks_csv": {
            "path": str(RANK_SOURCE.relative_to(REPO_ROOT)),
            "rows": len(rank_export),
            "sha256": rank_source_sha256,
        },
    },
    "outputs": {
        filename: {
            "rows": len(frame),
            "columns": frame.columns.tolist(),
            "sha256": sha256_file(OUTPUT_DIR / filename),
        }
        for filename, frame in output_frames.items()
    },
    "validation": {
        "full_transaction_rows": 1_754_155,
        "full_transaction_fraud_labels": 14_681,
        "overview_daily_rows": 183,
        "september_rank_rows": 287_873,
        "september_days": 30,
        "reconciled_policy_cohort_capacity_cases": 8,
        "scenario_lift_differences": [3, 10, 3, 9],
    },
    "publication": "Local-only outputs. Do not commit or redistribute pending publication clearance.",
}

manifest_path = OUTPUT_DIR / "PowerBI_Data_Manifest.json"
temporary_manifest = manifest_path.with_suffix(".json.tmp")
try:
    temporary_manifest.write_text(
        json.dumps(output_manifest, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary_manifest.replace(manifest_path)
finally:
    temporary_manifest.unlink(missing_ok=True)

output_inventory = pd.DataFrame([
    {
        "Artifact": filename,
        "Rows": len(frame),
        "Columns": len(frame.columns),
        "Size (KB)": round((OUTPUT_DIR / filename).stat().st_size / 1024, 1),
        "SHA-256": sha256_file(OUTPUT_DIR / filename)[:16] + "…",
    }
    for filename, frame in output_frames.items()
])
display(output_inventory)

assert (OUTPUT_DIR / "PowerBI_Data_Manifest.json").is_file()
assert len(pd.read_csv(OUTPUT_DIR / "Overview_Daily.csv")) == 183
assert len(pd.read_csv(OUTPUT_DIR / "DimDate.csv")) == 183
assert len(pd.read_csv(OUTPUT_DIR / "September_Ranks.csv")) == 287_873
assert len(pd.read_csv(OUTPUT_DIR / "September_Scenario_Lift.csv")) == 4
assert len(pd.read_csv(OUTPUT_DIR / "September_Queue_Reconciliation_QA.csv")) == 8

print(f"PASS · Local reporting outputs created in {OUTPUT_DIR.relative_to(REPO_ROOT)}.")
print("PASS · No raw or processed data files are written into tracked Git paths.")

## Final checkpoint

### What this notebook establishes

- The source still reconciles to 1,754,155 transactions and 14,681 retrospective fraud labels.
- The daily Overview table contains 183 rows and preserves the simulator's above/below-220 context.
- The September rank export contains 287,873 unique transactions across 30 days.
- All eight policy/cohort/capacity comparisons match the recorded P4 fraud-selected counts.
- The four fixed scenario differences are **+3, +10, +3 and +9**.
- The local Power BI import files are accompanied by a manifest of hashes and row counts.

### Next implementation increment

After this notebook completes on the owner's Windows environment, the next task is to establish the Power BI semantic model and DAX measures, then construct **Review Policy Comparison** first. The approved design remains unchanged.

**Important:** Power BI Desktop import, refresh, relationship behaviour and visual rendering still require verification in the owner's installed Power BI Desktop. This notebook does not claim those application checks have passed.